# Notebook 11 — LSTM Language Model for Text Generation (PyTorch)

*ML & NLP course — Data Trainers LLC — Axel Sirota*

## The story

Airbnb has over **200,000 listings**, each needing a human-written description. A new host faces a blank text box: what should they write? Low-quality descriptions translate directly to fewer bookings. Airbnb wants an **auto-complete tool** that generates plausible rental-style text given the host's first few words.

Your task: build a **neural language model** that predicts the next word given all previous words. Train it on real Airbnb descriptions. At inference, generate text word-by-word by repeatedly sampling from the model's predicted distribution.

This notebook is the capstone of your deep-learning journey in this course. Every concept you built before — embeddings, PyTorch Datasets, training loops, gradient clipping — comes together here.

## Learning objectives

By the end of this notebook you will be able to:

1. Frame text generation as **autoregressive next-token prediction**.
2. Explain RNN fundamentals: hidden state, unrolled computation, vanishing gradient problem.
3. Describe LSTM gates (forget, input, output, cell state) at a conceptual level.
4. Build training `(x, y)` pairs as **sliding windows** over token sequences.
5. Implement `Embedding → LSTM → Linear` in PyTorch with correct handling of `batch_first`, hidden state shapes, and padding.
6. Write a training loop with **teacher forcing** and track **perplexity** per epoch.
7. Implement inference-time generation: greedy, temperature sampling, and top-k sampling.
8. Carry hidden state across token generations for coherent output.
9. Discuss qualitative generation quality and tuning knobs.

## Prerequisites

- Notebooks 5–8 (embeddings, DataLoaders, training loops).
- PyTorch `nn.Module`, `Dataset`, `DataLoader`.

## How to run

- **Colab (recommended):** Runtime → Change runtime type → GPU (T4). Full training takes ~15–25 min.
- **CPU:** Works but slow. Reduce `EPOCHS = 3` and `CORPUS_SIZE = 5000`.

## Section 0 — Environment Setup

Install packages, import libraries, pin random seeds, detect GPU.

In [ ]:
# Install required packages (run this first on Google Colab)
# If running locally in a virtualenv that already has these, you can skip.
!pip install -q torch textblob pandas numpy matplotlib
!python -m textblob.download_corpora

In [ ]:
# Imports — grouped: stdlib -> visualization -> data -> model -> training
import math
import random
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from textblob import TextBlob

warnings.filterwarnings('ignore')

# ---- Hyperparameters (single source of truth — change here only) ----
SEED            = 42
CORPUS_SIZE     = 20_000     # subsample of training descriptions
VOCAB_MIN_FREQ  = 3          # drop words appearing fewer than this many times
EMBEDDING_DIM   = 128        # width of the embedding lookup table
HIDDEN_DIM      = 256        # LSTM hidden units per layer
NUM_LAYERS      = 2          # stacked LSTM layers
DROPOUT         = 0.3        # applied between LSTM layers
SEQ_LEN         = 20         # sliding window length for training pairs
BATCH_SIZE      = 64
EPOCHS          = 10
LR              = 1e-3
GRADIENT_CLIP   = 1.0        # CRITICAL: LSTMs explode without this
# Generation settings
MAX_NEW_TOKENS  = 40
TEMPERATURE     = 0.8
TOP_K           = 20

# ---- Set all random seeds for reproducibility ----
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- Device detection ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {device}")
if device.type == 'cuda':
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print("\nEnvironment setup complete.")

## Section 1 — What is a Language Model?

### The core idea

A **language model** assigns a probability to every possible next word given the words that came before:

$$P(w_t \mid w_1, w_2, \ldots, w_{t-1})$$

If you have this distribution, **generation is just sampling from it repeatedly**:

```
prompt = ["This", "rental", "offers"]
→ sample next word → ["This", "rental", "offers", "beautiful"]
→ sample next word → [..., "beautiful", "views"]
→ ... repeat until <eos>
```

This is called **autoregressive generation** — you feed your own previous output back as the next input.

### Perplexity

> *Perplexity measures how surprised the model is by the next word, on average.*

Formally it is the **exponent of cross-entropy loss**:

$$\text{PPL} = \exp(\text{CE loss})$$

Intuition: a perplexity of 100 means the model is as confused as if it had to choose uniformly among 100 equally likely next words. A perplexity of 10 means it has narrowed it down to ~10 plausible choices. **Lower is better.**

Initial PPL on a random model with vocab size `V` is approximately `V` — because the model distributes probability uniformly. As the model trains, PPL should drop from the thousands toward the tens.

In [ ]:
# Demo: perplexity intuition in two lines
import math

# Suppose vocab_size = 5000. Initial loss ≈ ln(5000).
vocab_size_demo = 5000
initial_loss = math.log(vocab_size_demo)   # cross-entropy of uniform distribution
initial_ppl  = math.exp(initial_loss)      # perplexity = exp(CE)

print(f"For vocab_size={vocab_size_demo}:")
print(f"  Expected initial CE loss : {initial_loss:.3f}")
print(f"  Expected initial PPL     : {initial_ppl:.1f}")
print()
print("After training well, a decent LSTM achieves PPL ~100-200 on Airbnb data.")
print("That means at each step the model has effectively narrowed it down to ~150 next words.")

## Section 2 — RNN and LSTM Theory

### The vanilla RNN

A recurrent network processes one token at a time, maintaining a **hidden state** that summarizes everything seen so far:

$$h_t = \tanh(W \cdot x_t + U \cdot h_{t-1} + b)$$

Unrolled over 4 tokens:

```
x_1 → [RNN] → h_1 → [RNN] → h_2 → [RNN] → h_3 → [RNN] → h_4
               x_2           x_3            x_4
```

The problem: **vanishing gradients**. During backpropagation through time, gradients are multiplied by the same weight matrix at every step. If any eigenvalue of that matrix is less than 1, gradients shrink exponentially. Information from 50 tokens ago effectively disappears.

### LSTM gates — the solution

LSTMs (Hochreiter & Schmidhuber, 1997) introduce a **cell state** `c_t` — a separate "memory highway" — controlled by learned gates:

| Gate | What it does |
|------|-------------|
| **Forget gate** `f_t` | Decides what to *drop* from cell state |
| **Input gate** `i_t` | Decides what *new info* to write to cell state |
| **Output gate** `o_t` | Decides what to *expose* as hidden state `h_t` |
| **Cell state** `c_t` | The long-term memory highway — survives many steps unchanged |

The key insight: the cell state flows through additive connections (not multiplicative ones), so gradients can travel long distances without vanishing.

```python
# PyTorch LSTM in one line:
lstm = nn.LSTM(input_size=128, hidden_size=256, num_layers=2,
               dropout=0.3, batch_first=True)
output, (h_n, c_n) = lstm(x)   # x: (B, T, 128) → output: (B, T, 256)
                                 # h_n, c_n: (num_layers, B, 256)
```

In [ ]:
# Demo: walk through nn.LSTM shapes — the most common source of confusion

# A 2-layer LSTM: input_size=100, hidden_size=128
demo_lstm = nn.LSTM(
    input_size=100,
    hidden_size=128,
    num_layers=2,
    batch_first=True   # <-- CRITICAL: expect (B, T, D), not (T, B, D)
)

# Create a fake batch: 4 sequences, 10 tokens each, 100-dim embedding
B, T, D_in = 4, 10, 100
fake_input = torch.randn(B, T, D_in)

# Forward pass — state=None means PyTorch auto-initializes h and c to zeros
output, (h_n, c_n) = demo_lstm(fake_input)

print("Input shape             :", fake_input.shape,  "  (B, T, input_size)")
print("Output shape            :", output.shape,      "  (B, T, hidden_size)")
print("h_n shape (hidden state):", h_n.shape,         "  (num_layers, B, hidden_size)")
print("c_n shape (cell state)  :", c_n.shape,         "  (num_layers, B, hidden_size)")
print()
print("Why batch_first=True?")
print("  PyTorch default is batch_LAST: input = (T, B, D) which is less intuitive.")
print("  With batch_first=True: input = (B, T, D) — same convention as every other layer.")

## Section 3 — Data Preparation

### Pipeline overview

1. Load Airbnb descriptions from Dropbox.
2. Tokenize with TextBlob, lowercase.
3. Build vocab with `MIN_FREQ=3`. Add special tokens: `<pad>`, `<unk>`, `<bos>`, `<eos>`.
4. Prepend `<bos>` and append `<eos>` to each description.
5. Create **sliding-window (x, y) pairs**: input = `tokens[i : i+SEQ_LEN]`, target = `tokens[i+1 : i+SEQ_LEN+1]`.
6. Wrap in a PyTorch `Dataset` + `DataLoader`.

The sliding-window approach means: **given 20 consecutive words, predict each of the 20 next words simultaneously**. This is equivalent to 20 separate single-step predictions but trained in parallel.

In [ ]:
# Load Airbnb rental descriptions from Dropbox
TRAIN_URL = 'https://www.dropbox.com/scl/fi/rbrynlq7871cshi0krftj/train_corpus_descriptions_airbnb.csv?rlkey=td1pfjgqjccap0xu9g4eliube&dl=1'
TEST_URL  = 'https://www.dropbox.com/scl/fi/eys05bzwwnhskadqh7aux/test_corpus_descriptions_airbnb.csv?rlkey=p1zuz90khh5t7dx3hkfba1dzm&dl=1'

# Load CSVs — these have no header; first column is the description text
train_df = pd.read_csv(TRAIN_URL, header=None, names=['description']).dropna()
test_df  = pd.read_csv(TEST_URL,  header=None, names=['description']).dropna()

print(f"Train descriptions : {len(train_df):,}")
print(f"Test descriptions  : {len(test_df):,}")

# Subsample to keep training time manageable
train_df = train_df.sample(n=min(CORPUS_SIZE, len(train_df)), random_state=SEED).reset_index(drop=True)
test_df  = test_df.sample(n=min(5000, len(test_df)),          random_state=SEED).reset_index(drop=True)

print(f"\nSubsampled train   : {len(train_df):,}")
print(f"Subsampled test    : {len(test_df):,}")
print("\nFirst description:")
print(train_df.iloc[0].description[:300])

In [ ]:
# Tokenize with TextBlob — lowercase; returns a list of word strings
def tokenize(text):
    """Lowercase tokenization via TextBlob. Returns list of word strings."""
    return [w.lower() for w in TextBlob(str(text)).words]

# Demo: tokenize one description
sample_tokens = tokenize(train_df.iloc[0].description)
print("First 20 tokens:", sample_tokens[:20])

# Tokenize all descriptions (this takes ~30–60s on 20K descriptions)
print("\nTokenizing all descriptions...")
train_tokens = [tokenize(t) for t in train_df['description']]
test_tokens  = [tokenize(t) for t in test_df['description']]

print(f"Tokenized {len(train_tokens):,} train descriptions")
lengths = [len(t) for t in train_tokens]
print(f"Avg tokens/description: {np.mean(lengths):.1f}  |  max: {max(lengths)}")

In [ ]:
# Build vocabulary with special tokens
# Special token ids are fixed: 0=<pad>, 1=<unk>, 2=<bos>, 3=<eos>

PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
BOS_TOKEN = '<bos>'   # beginning of sequence
EOS_TOKEN = '<eos>'   # end of sequence

SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

def build_vocab(all_token_lists, min_freq=3):
    """Build word->id and id->word mappings.
    Special tokens occupy ids 0-3. Regular words start at id 4.
    """
    # Count all tokens across all descriptions
    counts = Counter(tok for tokens in all_token_lists for tok in tokens)
    # Keep only words with sufficient frequency
    kept = [w for w, c in counts.most_common() if c >= min_freq]

    # Special tokens first (fixed ids), then vocabulary words
    word2id = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
    for w in kept:
        if w not in word2id:          # avoid overwriting special tokens
            word2id[w] = len(word2id)

    id2word = {i: w for w, i in word2id.items()}
    return word2id, id2word

word2id, id2word = build_vocab(train_tokens, min_freq=VOCAB_MIN_FREQ)
vocab_size = len(word2id)

# Retrieve special token ids for use throughout the notebook
PAD_IDX = word2id[PAD_TOKEN]   # 0
UNK_IDX = word2id[UNK_TOKEN]   # 1
BOS_IDX = word2id[BOS_TOKEN]   # 2
EOS_IDX = word2id[EOS_TOKEN]   # 3

print(f"Vocabulary size (min_freq={VOCAB_MIN_FREQ}): {vocab_size:,}")
print(f"Special token ids: PAD={PAD_IDX}, UNK={UNK_IDX}, BOS={BOS_IDX}, EOS={EOS_IDX}")
print(f"Sample vocab entries: {list(word2id.items())[4:14]}")

In [ ]:
# Encode a token list to integer ids, wrapping with <bos> and <eos>
def encode(tokens):
    """Convert token strings to integer ids. Unknown tokens map to UNK_IDX.
    Prepends BOS_IDX and appends EOS_IDX.
    """
    ids = [word2id.get(t, UNK_IDX) for t in tokens]
    return [BOS_IDX] + ids + [EOS_IDX]

# Demo: encode the first description
demo_ids = encode(train_tokens[0])
print("Tokens (first 10):", train_tokens[0][:10])
print("IDs    (first 12):", demo_ids[:12])
print("Decoded back     :", [id2word[i] for i in demo_ids[:12]])

# Encode all descriptions
train_ids = [encode(t) for t in train_tokens]
test_ids  = [encode(t) for t in test_tokens]
print(f"\nEncoded {len(train_ids):,} train sequences")

In [ ]:
# Demo: sliding-window pair generation
# Given a sequence of length L, we produce (L - SEQ_LEN) windows:
#   x = ids[i   : i + SEQ_LEN]   (input context)
#   y = ids[i+1 : i + SEQ_LEN+1] (targets, shifted right by 1)

def make_windows(ids, seq_len=SEQ_LEN):
    """Return a list of (x, y) integer-list pairs of length seq_len."""
    pairs = []
    for i in range(len(ids) - seq_len):
        x = ids[i     : i + seq_len]
        y = ids[i + 1 : i + seq_len + 1]
        pairs.append((x, y))
    return pairs

# Walk through a short example so the shift is clear
toy_ids  = demo_ids[:8]   # first 8 tokens of first description
toy_words = [id2word[i] for i in toy_ids]
print("Example sequence :", toy_words)
print("IDs              :", toy_ids)
print()
for x, y in make_windows(toy_ids, seq_len=4):
    x_words = [id2word[i] for i in x]
    y_words = [id2word[i] for i in y]
    print(f"  x={x_words}  →  y={y_words}")

### Lab 1 — Build `LMDataset`

Wrap the sliding-window logic into a PyTorch `Dataset`.

Your dataset should:

1. In `__init__`, call `make_windows` on every encoded sequence and collect all `(x, y)` pairs into a flat list.
2. In `__len__`, return the total number of pairs.
3. In `__getitem__(idx)`, return `(x_tensor, y_tensor)` where both are `torch.LongTensor` of shape `(SEQ_LEN,)`.

Then build a `DataLoader` with `batch_size=BATCH_SIZE, shuffle=True, num_workers=0`.

**Sanity check:** fetch one batch and confirm shapes are `(BATCH_SIZE, SEQ_LEN)` for both `x` and `y`.

In [ ]:
# Lab 1: implement LMDataset

class LMDataset(Dataset):
    def __init__(self, encoded_sequences, seq_len=SEQ_LEN):
        # 1. Collect every (x, y) pair from every encoded sequence.
        #    Call make_windows(seq, seq_len) for each sequence and extend a list.
        self.pairs = None  # YOUR CODE

    def __len__(self):
        # 2. Return the number of training pairs.
        return None  # YOUR CODE

    def __getitem__(self, idx):
        # 3. Return (x_tensor, y_tensor): both torch.long, shape (SEQ_LEN,)
        x, y = None, None  # YOUR CODE
        return x, y


# 4. Build datasets and DataLoaders for train and test
train_dataset = None  # YOUR CODE
test_dataset  = None  # YOUR CODE
train_loader  = None  # YOUR CODE (batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader   = None  # YOUR CODE (batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Sanity check — uncomment once implemented
# print(f"Train pairs: {len(train_dataset):,}")
# print(f"Test pairs : {len(test_dataset):,}")
# x_batch, y_batch = next(iter(train_loader))
# print(f"x_batch shape: {x_batch.shape}  (expected: ({BATCH_SIZE}, {SEQ_LEN}))")
# print(f"y_batch shape: {y_batch.shape}  (expected: ({BATCH_SIZE}, {SEQ_LEN}))")

## Section 4 — The LSTM Language Model

### Architecture

```
(B, T) integer ids
    │
    ▼  nn.Embedding(vocab_size, EMBEDDING_DIM)
(B, T, EMBEDDING_DIM)
    │
    ▼  nn.LSTM(EMBEDDING_DIM, HIDDEN_DIM, num_layers=2, batch_first=True)
(B, T, HIDDEN_DIM)
    │
    ▼  nn.Linear(HIDDEN_DIM, vocab_size)
(B, T, vocab_size)  ← logits for every position
```

Note: we produce logits at **every time step** simultaneously (thanks to teacher forcing at training time). The loss averages over all positions.

Key design choices:
- `padding_idx=PAD_IDX` in `nn.Embedding` — zero-vector for padding, no gradient through pads.
- `ignore_index=PAD_IDX` in `CrossEntropyLoss` — padded positions don't contribute to loss.
- `state` is `None` on the first call → PyTorch auto-initializes `h` and `c` to zeros.
- At inference, we carry state forward across generation steps.

In [ ]:
# Demo: instantiate the LSTM and verify the output shapes
# (we'll ask you to build the full model class in Lab 2)

demo_vocab  = 100
demo_emb    = nn.Embedding(demo_vocab, EMBEDDING_DIM, padding_idx=0)
demo_lstm   = nn.LSTM(EMBEDDING_DIM, HIDDEN_DIM, num_layers=NUM_LAYERS,
                      dropout=DROPOUT, batch_first=True)
demo_fc     = nn.Linear(HIDDEN_DIM, demo_vocab)

B_demo, T_demo = 4, SEQ_LEN
x_demo = torch.randint(0, demo_vocab, (B_demo, T_demo))

e_demo              = demo_emb(x_demo)                    # (B, T, E)
out_demo, state_demo = demo_lstm(e_demo)                  # (B, T, H), (h,c)
logits_demo         = demo_fc(out_demo)                   # (B, T, V)

print("x shape    :", x_demo.shape,       "  (B, T)")
print("emb shape  :", e_demo.shape,       "  (B, T, EMBEDDING_DIM)")
print("out shape  :", out_demo.shape,     "  (B, T, HIDDEN_DIM)")
print("h_n shape  :", state_demo[0].shape,"  (num_layers, B, HIDDEN_DIM)")
print("c_n shape  :", state_demo[1].shape,"  (num_layers, B, HIDDEN_DIM)")
print("logits     :", logits_demo.shape,  "  (B, T, vocab_size)")

### Lab 2 — Implement `LSTMLanguageModel`

Fill in the `__init__` and `forward` methods.

**`__init__` should create three layers:**
1. `nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)` — the lookup table.
2. `nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers, dropout=..., batch_first=True)` — the recurrent core. Only apply dropout if `num_layers > 1` (PyTorch ignores dropout on single-layer LSTMs).
3. `nn.Linear(hidden_dim, vocab_size)` — the output projection.

**`forward(self, x, state=None)` should:**
1. Look up embeddings → `(B, T, emb_dim)`.
2. Pass through LSTM with optional initial state → `(B, T, hidden_dim)` + new state.
3. Project to logits → `(B, T, vocab_size)`.
4. Return `(logits, state)` so the caller can carry state for generation.

In [ ]:
# Lab 2: implement LSTMLanguageModel

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM,
                 num_layers=NUM_LAYERS, dropout=DROPOUT, pad_idx=PAD_IDX):
        super().__init__()
        # 1. Embedding layer — padding_idx ensures pad tokens get zero vectors
        self.emb  = None  # YOUR CODE
        # 2. LSTM — batch_first=True so input is (B, T, D)
        #           only pass dropout if num_layers > 1 (else PyTorch warns)
        self.lstm = None  # YOUR CODE
        # 3. Output projection to vocabulary logits
        self.fc   = None  # YOUR CODE

    def forward(self, x, state=None):
        # x: (B, T) integer ids
        # state: tuple (h, c) or None (auto-initialized to zeros)

        # 4. Embedding lookup -> (B, T, emb_dim)
        e = None          # YOUR CODE
        # 5. LSTM forward -> out (B, T, hidden_dim) + new state (h, c)
        out, state = None, None  # YOUR CODE  (one line: out, state = self.lstm(e, state))
        # 6. Project to logits -> (B, T, vocab_size)
        logits = None     # YOUR CODE
        return logits, state


# Sanity check — uncomment once implemented
# model = LSTMLanguageModel(vocab_size).to(device)
# print(model)
# n_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {n_params:,}")
#
# x_dummy = torch.randint(0, vocab_size, (4, SEQ_LEN), device=device)
# logits_dummy, state_dummy = model(x_dummy)
# print(f"Output logits shape: {logits_dummy.shape}  (expected: (4, {SEQ_LEN}, {vocab_size}))")
# print(f"h_n shape: {state_dummy[0].shape}   c_n shape: {state_dummy[1].shape}")

## Section 5 — Training Loop + Perplexity

### Teacher forcing

At **training time** we always feed the **true previous tokens** as input (not the model's own predictions). This is called **teacher forcing**. It makes training faster and more stable — the model learns from clean examples without compounding its own errors.

With our sliding-window setup, teacher forcing is automatic: `x` is the window of true tokens, `y` is the same window shifted by 1.

### The reshape trick for CrossEntropyLoss

`nn.CrossEntropyLoss` expects:
- **logits**: shape `(N, C)` — N examples, C classes.
- **targets**: shape `(N,)` — N class indices.

Our logits are `(B, T, V)` and targets are `(B, T)`. The trick:

```python
loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
#               (B*T, V)                          (B*T,)
```

This treats every `(batch, time)` position as an independent example — equivalent to the sum over all positions.

### Gradient clipping

LSTMs can suffer from **exploding gradients** — the opposite of vanishing. If a gradient update is too large, the loss shoots to `nan`. The fix is one line:

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
```

This caps the global gradient norm to `GRADIENT_CLIP` (we use 1.0). Add it **after** `loss.backward()` and **before** `optimizer.step()`.

In [ ]:
# Demo: set up loss function and verify initial loss/perplexity

# CrossEntropyLoss with ignore_index=PAD_IDX so padding doesn't contaminate the loss
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

print("Loss function:", loss_fn)
print()
print(f"Vocab size = {vocab_size}")
print(f"Expected initial loss       ≈ ln({vocab_size}) = {math.log(vocab_size):.3f}")
print(f"Expected initial perplexity ≈ {math.exp(math.log(vocab_size)):.0f}")
print()
print("If your loss is MUCH higher than this, check:")
print("  - Are you passing logits (not softmax-ed output) to CrossEntropyLoss?")
print("  - Is vocab_size correct?")
print("  - Did you accidentally apply softmax in forward()?")

In [ ]:
# Demo: the reshape trick for CrossEntropyLoss
# This cell works even before Labs 1 and 2 are complete (uses demo shapes).

B_demo2 = 8
T_demo2 = SEQ_LEN
V_demo2 = vocab_size

fake_logits  = torch.randn(B_demo2, T_demo2, V_demo2)   # (B, T, V)
fake_targets = torch.randint(0, V_demo2, (B_demo2, T_demo2))  # (B, T)

# Reshape so CrossEntropyLoss sees (N, C) and (N,)
loss_demo = loss_fn(
    fake_logits.reshape(-1, V_demo2),    # (B*T, V)
    fake_targets.reshape(-1)             # (B*T,)
)
perplexity_demo = torch.exp(loss_demo)

print(f"logits shape before reshape : {fake_logits.shape}")
print(f"logits shape after reshape  : {fake_logits.reshape(-1, V_demo2).shape}")
print(f"targets shape after reshape : {fake_targets.reshape(-1).shape}")
print(f"Loss      : {loss_demo.item():.4f}")
print(f"Perplexity: {perplexity_demo.item():.2f}  (should be ≈ vocab_size for random logits)")

### Lab 3 — Write the Training Loop

Build the full training loop. It should:

1. Instantiate the model, loss function, and Adam optimizer.
2. For each epoch:
   a. Train — iterate over `train_loader`, apply the 6-step update (move → zero_grad → forward → loss → backward → **clip** → step).
   b. Evaluate — iterate over `test_loader` with `torch.no_grad()`, compute avg loss.
   c. Print: `Epoch N/E | train_loss | train_ppl | val_loss | val_ppl`.
3. After all epochs, plot both perplexity curves.

**Critical:** add `torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)` after `loss.backward()` and before `optimizer.step()`. Without this the loss can explode to `nan`.

Expected behavior: PPL should drop from ~vocab_size initially to ~100-300 after 10 epochs on a GPU.

In [ ]:
# Lab 3: full training loop
# Requires Labs 1 and 2 to be complete first.

# 1. Instantiate model, loss, optimizer
model     = None  # YOUR CODE: LSTMLanguageModel(vocab_size).to(device)
loss_fn   = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = None  # YOUR CODE: torch.optim.Adam(model.parameters(), lr=LR)

train_losses, val_losses = [], []
train_ppls,   val_ppls   = [], []

for epoch in range(EPOCHS):
    # ---- Training phase ----
    model.train()
    total_train_loss = 0.0
    n_train = 0
    for x, y in train_loader:
        # a. Move tensors to device
        x, y = None, None  # YOUR CODE

        # b. Clear accumulated gradients
        None  # YOUR CODE

        # c. Forward pass (ignore state — we reset each window)
        logits, _ = None, None  # YOUR CODE: model(x)

        # d. Compute loss — remember the reshape trick!
        loss = None  # YOUR CODE

        # e. Backpropagation
        None  # YOUR CODE

        # f. CRITICAL: clip gradients to prevent exploding loss
        None  # YOUR CODE: torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)

        # g. Update weights
        None  # YOUR CODE

        total_train_loss += loss.item()
        n_train += 1

    avg_train = total_train_loss / max(n_train, 1)
    train_losses.append(avg_train)
    train_ppls.append(math.exp(avg_train))

    # ---- Evaluation phase ----
    model.eval()
    total_val_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = None, None  # YOUR CODE
            logits, _ = None, None  # YOUR CODE: model(x)
            loss = None           # YOUR CODE
            total_val_loss += loss.item()
            n_val += 1

    avg_val = total_val_loss / max(n_val, 1)
    val_losses.append(avg_val)
    val_ppls.append(math.exp(avg_val))

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"train loss={avg_train:.4f} ppl={math.exp(avg_train):.1f} | "
          f"val loss={avg_val:.4f} ppl={math.exp(avg_val):.1f}")

In [ ]:
# Demo: plot training and validation perplexity curves
# (Run this after Lab 3 is complete)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(range(1, len(train_losses)+1), train_losses, 'b-o', label='train')
axes[0].plot(range(1, len(val_losses)+1),   val_losses,   'r-o', label='val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Cross-entropy Loss'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Perplexity curves
axes[1].plot(range(1, len(train_ppls)+1), train_ppls, 'b-o', label='train')
axes[1].plot(range(1, len(val_ppls)+1),   val_ppls,   'r-o', label='val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')
axes[1].set_title('Perplexity (lower is better)'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("If train PPL >> val PPL after many epochs: the model is underfitting (need more epochs or bigger model).")
print("If train PPL << val PPL: the model is overfitting (need more data or regularization).")

In [ ]:
# Save checkpoint so you don't need to retrain if the kernel restarts
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab_size': vocab_size,
    'word2id': word2id,
    'id2word': id2word,
    'train_ppls': train_ppls,
    'val_ppls': val_ppls,
}, 'lstm_lm_checkpoint.pt')
print("Checkpoint saved: lstm_lm_checkpoint.pt")

## Section 6 — Text Generation

### The inference loop

At inference time we generate word-by-word:

1. Start with `<bos>` (or a user-provided prompt encoded as token ids).
2. Feed through the model → get logits over the vocabulary.
3. Sample (or take argmax) the next token from the logits.
4. Append that token to our growing sequence.
5. Pass **only the last token** through the model, carrying the hidden state forward.
6. Repeat until `<eos>` or `max_new_tokens` is reached.

Step 5 is the key efficiency trick: instead of re-processing the entire sequence each step, we carry the LSTM state forward so it already encodes all previous context.

### Greedy vs. sampling vs. temperature

| Strategy | How | Trade-off |
|----------|-----|-----------|
| **Greedy** | Always pick `argmax(logits)` | Deterministic but repetitive |
| **Temperature sampling** | Divide logits by `T`, then sample | `T<1` → sharper / more confident; `T>1` → flatter / more random |
| **Top-k sampling** | Zero out all but top-k logits, renormalize, sample | Prevents picking from garbage tail |

The temperature `T` rescales the distribution before sampling:

```python
scaled_logits = logits / temperature
probs = torch.softmax(scaled_logits, dim=-1)
next_id = torch.multinomial(probs, num_samples=1)
```

`T=0` is mathematically equivalent to greedy (infinitely sharp). `T=1` is sampling from the model as-is. `T>1` makes output more random/creative.

In [ ]:
# Demo: complete generate() function with temperature and top-k sampling
# Study this carefully — Lab 4 asks you to re-implement it.

@torch.no_grad()
def generate_demo(model, prompt_words, max_new_tokens=MAX_NEW_TOKENS,
                  temperature=TEMPERATURE, top_k=None):
    """Generate text autoregressively.

    Args:
        model: trained LSTMLanguageModel
        prompt_words: list of word strings (prompt)
        max_new_tokens: how many tokens to generate
        temperature: > 1 = more random, < 1 = more deterministic, 0 = greedy
        top_k: if set, only sample from the top-k most likely tokens
    Returns:
        Generated text as a string.
    """
    model.eval()

    # Encode the prompt and add BOS
    prompt_ids = [BOS_IDX] + [word2id.get(w.lower(), UNK_IDX) for w in prompt_words]
    ids = torch.tensor(prompt_ids, device=device).unsqueeze(0)  # (1, T_prompt)

    # Warm up: run the prompt through the model to get the initial hidden state
    logits, state = model(ids, state=None)

    generated = list(prompt_ids)

    for _ in range(max_new_tokens):
        # Only look at the logits for the LAST position
        last_logits = logits[:, -1, :]    # (1, vocab_size)

        if temperature <= 0:
            # Greedy: just take argmax
            next_id = last_logits.argmax(dim=-1)  # (1,)
        else:
            # Temperature scaling
            scaled = last_logits / temperature     # (1, V)

            if top_k is not None and top_k > 0:
                # Top-k: zero out all but top-k logits, then renormalize
                top_vals, top_idx = torch.topk(scaled, top_k, dim=-1)  # (1, k)
                # Scatter top_k probs back into a full-vocab distribution
                probs = torch.zeros_like(scaled).scatter_(
                    1, top_idx, torch.softmax(top_vals, dim=-1)
                )  # (1, V)
            else:
                # Standard temperature sampling over full vocab
                probs = torch.softmax(scaled, dim=-1)  # (1, V)

            # Sample from the distribution
            next_id = torch.multinomial(probs, num_samples=1).squeeze(1)  # (1,)

        next_id_val = next_id.item()
        generated.append(next_id_val)

        if next_id_val == EOS_IDX:
            break

        # Carry state forward: only pass the last generated token
        next_token = next_id.unsqueeze(0)        # (1, 1)
        logits, state = model(next_token, state)

    # Decode: drop BOS, stop at EOS
    words = []
    for i in generated[1:]:   # skip BOS
        if i == EOS_IDX:
            break
        words.append(id2word.get(i, UNK_TOKEN))
    return ' '.join(words)


# Quick test (requires trained model from Lab 3)
# text = generate_demo(model, ['This', 'rental', 'offers'], temperature=0.8, top_k=20)
# print("Generated:", text)
print("Demo generate() function defined. Uncomment the test after completing Lab 3.")

### Lab 4 — Implement `generate()`

Re-implement the generation function from scratch using the placeholders below. The key steps:

1. Encode the prompt words to ids, prepend `BOS_IDX`.
2. Warm up the model on the full prompt → get initial `logits` and `state`.
3. Loop `max_new_tokens` times:
   a. Extract last-position logits: `last_logits = logits[:, -1, :]`.
   b. Apply temperature scaling if `temperature > 0`.
   c. If `top_k` is set: zero out all but top-k logits, renormalize with softmax.
   d. Sample with `torch.multinomial(probs, 1)` or use `argmax` for greedy.
   e. Break if `EOS_IDX` is sampled.
   f. Carry state: call `model(next_token, state)` with only the last token.
4. Decode the generated ids to words, skipping BOS/EOS.

In [ ]:
# Lab 4: implement generate()

@torch.no_grad()
def generate(model, prompt_words, max_new_tokens=MAX_NEW_TOKENS,
             temperature=TEMPERATURE, top_k=TOP_K):
    """Generate text autoregressively using the trained LSTM language model."""
    model.eval()

    # 1. Encode the prompt and prepend BOS
    prompt_ids = None  # YOUR CODE: [BOS_IDX] + [word2id.get(...) for w in prompt_words]
    ids = None         # YOUR CODE: torch.tensor(prompt_ids, device=device).unsqueeze(0)  # (1, T)

    # 2. Warm up: run the full prompt through the model to get initial state
    logits, state = None, None  # YOUR CODE: model(ids, state=None)

    generated = list(prompt_ids)

    for _ in range(max_new_tokens):
        # 3a. Extract last-position logits — shape (1, V)
        last_logits = None  # YOUR CODE: logits[:, -1, :]

        if temperature <= 0:
            # 3b-greedy. Take argmax
            next_id = None  # YOUR CODE: last_logits.argmax(dim=-1)
        else:
            # 3b-temperature. Scale logits
            scaled = None  # YOUR CODE: last_logits / temperature

            if top_k is not None and top_k > 0:
                # 3c. Top-k masking
                top_vals, top_idx = None, None  # YOUR CODE: torch.topk(scaled, top_k, dim=-1)
                probs = torch.zeros_like(scaled).scatter_(
                    1, top_idx, torch.softmax(top_vals, dim=-1)
                )
            else:
                probs = None  # YOUR CODE: torch.softmax(scaled, dim=-1)

            # 3d. Sample from probability distribution
            next_id = None  # YOUR CODE: torch.multinomial(probs, num_samples=1).squeeze(1)

        next_id_val = next_id.item()
        generated.append(next_id_val)

        # 3e. Stop if EOS is generated
        if next_id_val == EOS_IDX:
            break

        # 3f. Carry state forward — only pass the single new token
        next_token = None        # YOUR CODE: next_id.unsqueeze(0)  # (1, 1)
        logits, state = None, None  # YOUR CODE: model(next_token, state)

    # 4. Decode to words, skip BOS and stop at EOS
    words = []
    for i in generated[1:]:  # skip leading BOS
        if i == EOS_IDX:
            break
        words.append(id2word.get(i, UNK_TOKEN))
    return ' '.join(words)


# Verification — uncomment once Lab 4 is complete
# prompt = ['This', 'rental', 'offers']
# print("Generated (T=0.8, top-k=20):")
# print(generate(model, prompt, temperature=0.8, top_k=20))
print("generate() skeleton defined. Fill in placeholders and uncomment the verification block.")

In [ ]:
# Demo: compare greedy vs. temperature vs. top-k generation
# (Run after Labs 3 and 4 are complete)

prompts = [
    ['This', 'rental', 'offers'],
    ['The', 'apartment', 'is'],
    ['Located', 'in', 'the'],
]

strategies = [
    {'label': 'Greedy (T=0)',         'temperature': 0.0,  'top_k': None},
    {'label': 'Temperature T=0.5',    'temperature': 0.5,  'top_k': None},
    {'label': 'Temperature T=1.0',    'temperature': 1.0,  'top_k': None},
    {'label': 'Top-k=20 T=0.8',       'temperature': 0.8,  'top_k': 20},
]

# Uncomment once Labs 3 and 4 are complete:
# for prompt in prompts:
#     print(f"\nPrompt: '{' '.join(prompt)}'")
#     print('-' * 60)
#     for s in strategies:
#         text = generate(model, prompt, temperature=s['temperature'], top_k=s['top_k'])
#         print(f"[{s['label']:25s}]: {text[:80]}")

print("Comparison cell ready. Uncomment after completing Labs 3 and 4.")

## Section 7 — Ablation & Model Analysis

An **ablation study** removes or changes one component at a time to understand its contribution. Below we explore three axes:

1. **Model size:** tiny (hidden=64) vs. standard (hidden=256) — does capacity help?
2. **Stacked layers:** 1 vs. 2 LSTM layers — does depth help?
3. **Temperature effect:** generate 5 samples from the same prompt with T=0.5, 1.0, 1.5.

These are quick experiments — don't run a full training cycle for each, just a few epochs.

In [ ]:
# Demo: temperature effect — same prompt, different temperatures
# (Requires trained model from Lab 3)

demo_prompt = ['The', 'space', 'features']
temperatures = [0.5, 1.0, 1.5]

# Uncomment after Labs 3 and 4:
# print(f"Prompt: '{' '.join(demo_prompt)}'\n")
# for t in temperatures:
#     sample = generate(model, demo_prompt, temperature=t, top_k=None, max_new_tokens=30)
#     print(f"T={t:.1f}: {sample}")
#     print()

print("Temperature sweep cell ready. Uncomment after completing Labs 3 and 4.")
print()
print("Expected observations:")
print("  T=0.5 → more focused, repetitive, 'safe' language")
print("  T=1.0 → varied, balanced")
print("  T=1.5 → creative but sometimes incoherent")

### Lab 5 — Train a tiny variant and compare

Train a smaller version of the model — `hidden_dim=64, num_layers=1, epochs=3` — and compare its final validation perplexity to the full model.

Steps:
1. Instantiate a small model with `LSTMLanguageModel(vocab_size, hidden_dim=64, num_layers=1)`.
2. Train for 3 epochs using the same training loop logic from Lab 3.
3. Compute final validation perplexity.
4. Generate 2 samples from prompt `['This', 'rental', 'offers']`.
5. Print: `Small model val PPL: X.X | Full model val PPL: Y.Y`.

In [ ]:
# Lab 5: tiny model ablation

# 1. Instantiate small model
small_model = None  # YOUR CODE: LSTMLanguageModel(vocab_size, hidden_dim=64, num_layers=1).to(device)
small_optimizer = None  # YOUR CODE
small_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

SMALL_EPOCHS = 3
small_val_ppls = []

# 2. Train for 3 epochs
for epoch in range(SMALL_EPOCHS):
    small_model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        small_optimizer.zero_grad()
        logits, _ = None, None  # YOUR CODE
        loss = None             # YOUR CODE
        loss.backward()
        torch.nn.utils.clip_grad_norm_(small_model.parameters(), GRADIENT_CLIP)
        small_optimizer.step()

    # 3. Evaluate
    small_model.eval()
    total_v, n_v = 0.0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits, _ = None, None  # YOUR CODE
            loss = None             # YOUR CODE
            total_v += loss.item(); n_v += 1
    v_ppl = math.exp(total_v / max(n_v, 1))
    small_val_ppls.append(v_ppl)
    print(f"Small model epoch {epoch+1}/{SMALL_EPOCHS} | val ppl={v_ppl:.1f}")

# 4. Generate samples
# sample1 = generate(small_model, ['This', 'rental', 'offers'])  # YOUR CODE: uncomment
# sample2 = generate(small_model, ['The', 'apartment', 'has'])
# print("\nSmall model samples:")
# print(" 1:", sample1)
# print(" 2:", sample2)

# 5. Compare
# full_final_ppl = val_ppls[-1] if val_ppls else float('nan')
# print(f"\nSmall model (hidden=64, 1-layer, 3 epochs) val PPL: {small_val_ppls[-1]:.1f}")
# print(f"Full  model (hidden=256, 2-layer, {EPOCHS} epochs) val PPL: {full_final_ppl:.1f}")
print("Lab 5 skeleton ready. Fill in YOUR CODE placeholders to run the ablation.")

### Optional Lab — Sweep generation strategies

Generate 3 samples from 3 different prompts using each of: greedy, T=0.8/top-k=20, T=1.2/top-k=50. Print them side by side in a table format.

Discuss: which strategy produces the best-quality Airbnb descriptions in your opinion? What makes them "good"?

In [ ]:
# Optional lab: generation strategy sweep

opt_prompts = [
    ['Enjoy', 'your', 'stay'],
    ['The', 'bedroom', 'has'],
    ['Guests', 'have', 'access'],
]

opt_strategies = [
    {'label': 'Greedy',          'temperature': 0.0, 'top_k': None},
    {'label': 'T=0.8, top-k=20', 'temperature': 0.8, 'top_k': 20},
    {'label': 'T=1.2, top-k=50', 'temperature': 1.2, 'top_k': 50},
]

# YOUR CODE: loop over prompts and strategies, print results in a readable format
# Hint: the table structure already works with the generate() function from Lab 4.

## What you learned

- **Autoregressive language modeling** — framing generation as repeated next-token prediction.
- **Perplexity** — `exp(cross-entropy)`, the canonical metric for language models.
- **LSTM gating** — how forget/input/output gates solve the vanishing gradient problem that breaks vanilla RNNs.
- **Teacher forcing** — feeding true previous tokens during training for stable convergence.
- **Gradient clipping** — the essential guard against LSTM gradient explosion.
- **Hidden state carry-over** — the inference trick that makes generation efficient (don't re-process all prior tokens).
- **Temperature and top-k sampling** — two knobs that control the creativity/coherence trade-off.

### Limitations of LSTM language models

- **Long-range dependencies drift.** The hidden state must compress all prior context into a fixed-size vector. Beyond ~50 tokens the signal from early context weakens.
- **Sequential by design.** Each step depends on the previous, so training can't be fully parallelized over the time dimension.
- **Transformers fix both.** Attention looks directly at every previous token (no compression), and the computation over positions is fully parallel. That's why GPT, BERT, and every modern LLM is transformer-based.

### Self-check quiz

1. Initial loss on random init for `vocab_size=5000` should be approximately what value? *(Answer: `ln(5000) ≈ 8.52`. If much higher, something is wrong in your setup.)*
2. Your generation starts repeating `"bedroom bedroom bedroom bedroom"`. What's happening and which knob fixes it?
3. Why does `ignore_index=PAD_IDX` matter for the loss function?
4. You trained for 10 epochs and perplexity is still 800. Name 3 likely bugs.

### Next up

**Notebook 12 — Text Generation with GPT-2.** You will fine-tune `gpt2` (124M params) on the exact same Airbnb dataset and compare: perplexity, coherence length, training time, and generation quality. Spoiler: GPT-2 wins, but it needs 40× more parameters and a GPU — the LSTM still shines in low-resource settings.